
# 05 XAI + Fairness（Phase 5）

这个 notebook 对应 [Agent.md](/Users/kingjason/资源/credit%20visable/Agent.md) 里的 **Phase 5: Explainability and Fairness**。

当前版本默认消费 `Phase 4` 已落盘的 validation artifact，不直接重训模型。核心目标有两条：

- 解释候选模型及其 proxy uplift 的主要来源
- 对 `age` / `marital status` 做 fairness baseline，并对 `region` / `city` / `organization` / `occupation` 做 governance-sensitive grouped review

重要约束：

- `shap` 在当前环境里虽然可发现，但不保证可导入成功
- 若候选 advanced backend 是 `xgboost`，优先用 `pred_contribs=True` 生成 TreeSHAP 风格贡献值
- 其余情况回退到 permutation importance
- 本 notebook 产出的是 **grouped diagnostic**，不是 production-ready fairness audit



## 1. 运行参数与辅助函数

这里统一声明 `Phase 5` 的路径、阈值、分组规则与绘图工具。所有解释与 fairness 结果都围绕同一 validation split 展开。


In [ ]:

from pathlib import Path
import json
import sys
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse

root = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "pyproject.toml").exists()),
    Path.cwd(),
)
src = root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

warnings.filterwarnings(
    "ignore",
    message="Pandas requires version '1.3.6' or newer of 'bottleneck'",
)

from credit_visable.config import load_settings
from credit_visable.explainability import (
    compute_permutation_importance_summary,
    compute_xgboost_contribution_summary,
    compute_xgboost_local_explanations,
    get_explainability_runtime_status,
    select_local_explanation_rows,
)
from credit_visable.features import FEATURE_SET_NAMES as SUPPORTED_FEATURE_SET_NAMES
from credit_visable.governance import (
    build_group_fairness_summary,
    collapse_rare_categories,
    derive_age_band_from_days_birth,
)
from credit_visable.utils.paths import get_paths

settings = load_settings()
paths = get_paths()

FEATURE_SET_NAMES = list(SUPPORTED_FEATURE_SET_NAMES)
PREPROCESSING_OUTPUT_ROOT = paths.data_processed / "preprocessing"
BASELINE_OUTPUT_ROOT = paths.data_processed / "modeling_baseline"
ADVANCED_OUTPUT_ROOT = paths.data_processed / "modeling_advanced"
PHASE_5_OUTPUT_ROOT = paths.data_processed / "xai_fairness"
FIGURE_OUTPUT_DIR = paths.reports_figures
APPLICATION_TRAIN_PATH = paths.data_raw / settings.expected_tables["application_train"]

OPERATING_THRESHOLD = 0.50
EXPLAIN_SAMPLE_SIZE = 2000
TOP_N_FEATURES = 20
TOP_N_CATEGORIES = 8
LOCAL_CASES = 3
RANDOM_STATE = settings.random_state

AGE_BAND_ORDER = ["[20,25)", "[25,35)", "[35,45)", "[45,55)", "[55,65)", "[65,70)", "Missing"]
GROUP_SPECS = [
    {
        "protected_attribute": "age_band",
        "source_column": "DAYS_BIRTH",
        "group_column": "age_band",
        "kind": "age_band",
    },
    {
        "protected_attribute": "family_status_group",
        "source_column": "NAME_FAMILY_STATUS",
        "group_column": "family_status_group",
        "kind": "identity",
    },
    {
        "protected_attribute": "region_rating_group",
        "source_column": "REGION_RATING_CLIENT",
        "group_column": "region_rating_group",
        "kind": "identity",
    },
    {
        "protected_attribute": "city_work_mismatch_group",
        "source_column": "REG_CITY_NOT_WORK_CITY",
        "group_column": "city_work_mismatch_group",
        "kind": "identity",
    },
    {
        "protected_attribute": "organization_group",
        "source_column": "ORGANIZATION_TYPE",
        "group_column": "organization_group",
        "kind": "top_categories",
        "top_n": TOP_N_CATEGORIES,
    },
    {
        "protected_attribute": "occupation_group",
        "source_column": "OCCUPATION_TYPE",
        "group_column": "occupation_group",
        "kind": "top_categories",
        "top_n": TOP_N_CATEGORIES,
    },
]
REVIEW_COLUMNS = [
    settings.id_column,
    "DAYS_BIRTH",
    "NAME_FAMILY_STATUS",
    "REGION_RATING_CLIENT",
    "REG_CITY_NOT_WORK_CITY",
    "ORGANIZATION_TYPE",
    "OCCUPATION_TYPE",
]

PHASE_5_READY = False
phase5_load_ok = False
phase5_selection_ok = False
phase5_explainability_ok = False
phase5_fairness_ok = False
runtime_status = {}
artifacts_by_set = {}
review_frames_by_set = {}
phase4_comparison_metrics = pd.DataFrame()
selection_payload = {}
candidate_model_selection_payload = {}
explainability_outputs_by_set = {}
local_case_selection = pd.DataFrame()
local_case_outputs_by_set = {}
proxy_uplift_summary = pd.DataFrame()
group_fairness_summary = pd.DataFrame()
group_gap_summary = pd.DataFrame()
validation_review_frame = pd.DataFrame()
figure_paths = {}
summary_payload = {}


def build_preprocessing_files(output_dir: Path) -> dict[str, Path]:
    return {
        "X_valid": output_dir / "X_valid.npz",
        "valid_meta": output_dir / "valid_meta.csv",
        "feature_names": output_dir / "feature_names.csv",
        "manifest": output_dir / "manifest.json",
    }


def build_baseline_files(output_dir: Path) -> dict[str, Path]:
    return {
        "metrics": output_dir / "metrics.json",
        "validation_scores": output_dir / "validation_scores.csv",
        "model": output_dir / "model.joblib",
    }


def build_advanced_files(output_dir: Path) -> dict[str, Path]:
    return {
        "metrics": output_dir / "metrics.json",
        "validation_scores": output_dir / "validation_scores.csv",
        "model": output_dir / "advanced_model.joblib",
    }


def _matrix_density(matrix: sparse.spmatrix) -> float:
    total_cells = matrix.shape[0] * matrix.shape[1]
    if total_cells == 0:
        return 0.0
    return float(matrix.nnz / total_cells)


def _to_builtin(value):
    if isinstance(value, dict):
        return {str(key): _to_builtin(item) for key, item in value.items()}
    if isinstance(value, list):
        return [_to_builtin(item) for item in value]
    if isinstance(value, tuple):
        return [_to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return [_to_builtin(item) for item in value.tolist()]
    if isinstance(value, (np.floating, float)):
        if not np.isfinite(value):
            return None
        return float(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    if pd.isna(value):
        return None
    return value


def _sort_group_summary(summary_frame: pd.DataFrame, protected_attribute: str) -> pd.DataFrame:
    if summary_frame.empty:
        return summary_frame.copy()

    ordered = summary_frame.copy()
    if protected_attribute == "age_band":
        ordered["group"] = pd.Categorical(
            ordered["group"],
            categories=AGE_BAND_ORDER,
            ordered=True,
        )
        ordered = ordered.sort_values("group")
        ordered["group"] = ordered["group"].astype(str)
        return ordered.reset_index(drop=True)

    return ordered.sort_values(["count", "group"], ascending=[False, True]).reset_index(drop=True)


def _feature_label_from_local_row(row: pd.Series) -> str:
    encoded_value = row.get("encoded_value")
    if pd.notna(encoded_value) and str(encoded_value) not in {"", "None"}:
        return f"{row['raw_feature_name']}={encoded_value}"
    return str(row["raw_feature_name"])


def save_horizontal_bar_chart(
    frame: pd.DataFrame,
    value_column: str,
    label_column: str,
    title: str,
    path: Path,
    top_n: int | None = None,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plot_frame = frame.copy()
    if top_n is not None:
        plot_frame = plot_frame.head(top_n)

    fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * max(len(plot_frame), 1))))
    if plot_frame.empty:
        ax.text(0.5, 0.5, "No data available", ha="center", va="center")
        ax.axis("off")
    else:
        plot_frame = plot_frame.sort_values(value_column, ascending=True)
        colors = np.where(plot_frame[value_column] >= 0, "#4C78A8", "#E45756")
        ax.barh(plot_frame[label_column].astype(str), plot_frame[value_column], color=colors)
        ax.set_xlabel(value_column.replace("_", " ").title())
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def save_group_metric_chart(
    summary_frame: pd.DataFrame,
    protected_attribute: str,
    metric_column: str,
    title: str,
    path: Path,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plot_frame = summary_frame.loc[
        summary_frame["protected_attribute"] == protected_attribute
    ].copy()
    plot_frame = _sort_group_summary(plot_frame, protected_attribute)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    if plot_frame.empty:
        ax.text(0.5, 0.5, "No data available", ha="center", va="center")
        ax.axis("off")
    else:
        ax.bar(plot_frame["group"].astype(str), plot_frame[metric_column], color="#4C78A8")
        ax.set_ylabel(metric_column.replace("_", " ").title())
        ax.tick_params(axis="x", rotation=35)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def save_histogram(frame: pd.DataFrame, column: str, title: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    if frame.empty or column not in frame.columns:
        ax.text(0.5, 0.5, "No data available", ha="center", va="center")
        ax.axis("off")
    else:
        ax.hist(frame[column].dropna(), bins=30, color="#4C78A8", edgecolor="white", alpha=0.85)
        ax.set_xlabel(column.replace("_", " ").title())
        ax.set_ylabel("Count")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def save_local_case_chart(
    local_frame: pd.DataFrame,
    case_role: str,
    title: str,
    path: Path,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plot_frame = local_frame.loc[local_frame["case_role"] == case_role].copy() if not local_frame.empty else pd.DataFrame()
    fig, ax = plt.subplots(figsize=(9, max(4, 0.4 * max(len(plot_frame), 1))))
    if plot_frame.empty:
        ax.text(
            0.5,
            0.5,
            "Local TreeSHAP-style contributions are unavailable for this candidate model.",
            ha="center",
            va="center",
            wrap=True,
        )
        ax.axis("off")
    else:
        plot_frame["feature_label"] = plot_frame.apply(_feature_label_from_local_row, axis=1)
        plot_frame = plot_frame.sort_values("contribution", ascending=True)
        colors = np.where(plot_frame["contribution"] >= 0, "#E45756", "#4C78A8")
        ax.barh(plot_frame["feature_label"], plot_frame["contribution"], color=colors)
        ax.set_xlabel("Contribution to model margin")
        if "candidate_predicted_pd" in plot_frame.columns:
            title = f"{title} | candidate PD = {plot_frame.iloc[0]['candidate_predicted_pd']:.4f}"
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def select_candidate_payload(comparison_metrics: pd.DataFrame) -> dict[str, object]:
    ranked = comparison_metrics.copy()
    ranked["model_family_preference"] = ranked["model_family"].map(
        {"advanced": 1, "logistic": 0}
    ).fillna(0)
    ranked["feature_set_preference"] = ranked["feature_set"].map(
        {"traditional_plus_proxy": 1, "traditional_core": 0}
    ).fillna(0)
    ranked = ranked.sort_values(
        ["roc_auc", "average_precision", "model_family_preference", "feature_set_preference"],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)

    candidate_model = ranked.iloc[0].to_dict()
    matched_core = ranked.loc[
        (ranked["model_family"] == candidate_model["model_family"])
        & (ranked["feature_set"] == "traditional_core")
    ]
    matched_proxy = ranked.loc[
        (ranked["model_family"] == candidate_model["model_family"])
        & (ranked["feature_set"] == "traditional_plus_proxy")
    ]

    matched_core_payload = matched_core.iloc[0].to_dict() if not matched_core.empty else None
    matched_proxy_payload = matched_proxy.iloc[0].to_dict() if not matched_proxy.empty else None
    if matched_core_payload is not None and matched_proxy_payload is not None:
        proxy_audit_pair_family = candidate_model["model_family"]
        proxy_audit_pair_reason = "same_as_candidate_model_family"
    else:
        proxy_audit_pair_family = "logistic"
        proxy_audit_pair_reason = "fallback_to_logistic_pair"
        matched_core_payload = ranked.loc[
            (ranked["model_family"] == "logistic")
            & (ranked["feature_set"] == "traditional_core")
        ].iloc[0].to_dict()
        matched_proxy_payload = ranked.loc[
            (ranked["model_family"] == "logistic")
            & (ranked["feature_set"] == "traditional_plus_proxy")
        ].iloc[0].to_dict()

    return {
        "ranked_comparison": ranked,
        "candidate_model": candidate_model,
        "matched_core_comparator": matched_core_payload,
        "matched_proxy_comparator": matched_proxy_payload,
        "proxy_audit_pair_family": proxy_audit_pair_family,
        "proxy_audit_pair_reason": proxy_audit_pair_reason,
    }


def build_group_gap_summary(summary_frame: pd.DataFrame) -> pd.DataFrame:
    if summary_frame.empty:
        return pd.DataFrame()

    rows = []
    for protected_attribute, subset in summary_frame.groupby("protected_attribute"):
        rows.append(
            {
                "protected_attribute": protected_attribute,
                "max_actual_default_rate_gap": float(
                    subset["actual_default_rate"].max() - subset["actual_default_rate"].min()
                ),
                "max_mean_predicted_pd_gap": float(
                    subset["mean_predicted_pd"].max() - subset["mean_predicted_pd"].min()
                ),
                "max_approval_rate_gap": float(
                    subset["approval_rate"].max() - subset["approval_rate"].min()
                ),
                "min_approval_rate_vs_best_group": float(
                    subset["approval_rate_vs_best_group"].min()
                ),
            }
        )

    return pd.DataFrame(rows).sort_values(
        ["max_approval_rate_gap", "max_mean_predicted_pd_gap"],
        ascending=[False, False],
    ).reset_index(drop=True)


print(f"Application train path: {APPLICATION_TRAIN_PATH}")
print(f"Preprocessing output root: {PREPROCESSING_OUTPUT_ROOT}")
print(f"Baseline output root: {BASELINE_OUTPUT_ROOT}")
print(f"Advanced output root: {ADVANCED_OUTPUT_ROOT}")
print(f"Phase 5 output root: {PHASE_5_OUTPUT_ROOT}")
print(f"Figure output dir: {FIGURE_OUTPUT_DIR}")
print(f"Feature sets: {FEATURE_SET_NAMES}")



## 2. Readiness gate

`Phase 5` 依赖双 feature-set 的 preprocessing / baseline / advanced artifact，以及 `application_train.csv` 的 fairness / governance 字段。这里还会单独记录 explainability runtime 状态，避免把“模块可发现”误写成“运行可用”。


In [ ]:

required_root_files = {
    "application_train": APPLICATION_TRAIN_PATH,
    "phase4_comparison_metrics": ADVANCED_OUTPUT_ROOT / "comparison_metrics.csv",
}
required_files_by_set = {
    feature_set_name: {
        "preprocessing": build_preprocessing_files(PREPROCESSING_OUTPUT_ROOT / feature_set_name),
        "baseline": build_baseline_files(BASELINE_OUTPUT_ROOT / feature_set_name),
        "advanced": build_advanced_files(ADVANCED_OUTPUT_ROOT / feature_set_name),
    }
    for feature_set_name in FEATURE_SET_NAMES
}

readiness_rows = [
    {
        "scope": "root",
        "feature_set": "all",
        "stage": "phase5_root",
        "artifact": artifact_name,
        "path": str(path),
        "exists": path.exists(),
    }
    for artifact_name, path in required_root_files.items()
]
for feature_set_name, stage_map in required_files_by_set.items():
    for stage_name, artifact_map in stage_map.items():
        readiness_rows.extend(
            [
                {
                    "scope": stage_name,
                    "feature_set": feature_set_name,
                    "stage": stage_name,
                    "artifact": artifact_name,
                    "path": str(path),
                    "exists": path.exists(),
                }
                for artifact_name, path in artifact_map.items()
            ]
        )

readiness_frame = pd.DataFrame(readiness_rows)
runtime_status = get_explainability_runtime_status()
PHASE_5_READY = bool(readiness_frame["exists"].all())

display(readiness_frame)
display(pd.DataFrame([runtime_status]))
print(f"Phase 5 ready: {PHASE_5_READY}")

if not PHASE_5_READY:
    display(readiness_frame.loc[~readiness_frame["exists"]].reset_index(drop=True))
    print(
        "Phase 5 blocked. 请先完成双 feature-set 的 Phase 2 / Phase 3 / Phase 4 标准产物，并确认 application_train.csv 可用。"
    )
elif runtime_status["shap_module_found"] and not runtime_status["shap_import_ok"]:
    print(
        "注意：shap 模块可发现，但当前环境无法成功导入。Phase 5 会优先尝试 xgboost pred_contribs，并在必要时回退到 permutation importance。"
    )



## 3. 加载并对齐双 feature-set artifact

这里会统一读取两套 preprocessing、baseline、advanced 产物，并按 `SK_ID_CURR` 回连 `application_train.csv` 的 fairness / governance 字段。只有当 validation 主键与分数结果一一对应时，后面的解释与 grouped review 才有意义。


In [ ]:

if not PHASE_5_READY:
    print("加载步骤已跳过，因为 Phase 5 依赖产物不完整。")
else:
    artifacts_by_set = {}
    review_frames_by_set = {}
    phase4_comparison_metrics = pd.DataFrame()
    load_summary_rows = []
    validation_errors = []
    reference_valid_meta = None
    advanced_backend_reference = None

    phase4_comparison_metrics = pd.read_csv(required_root_files["phase4_comparison_metrics"])
    review_source_frame = pd.read_csv(APPLICATION_TRAIN_PATH, usecols=REVIEW_COLUMNS)
    review_source_frame = review_source_frame.drop_duplicates(subset=[settings.id_column])

    for feature_set_name in FEATURE_SET_NAMES:
        preprocessing_files = required_files_by_set[feature_set_name]["preprocessing"]
        baseline_files = required_files_by_set[feature_set_name]["baseline"]
        advanced_files = required_files_by_set[feature_set_name]["advanced"]

        X_valid = sparse.load_npz(preprocessing_files["X_valid"])
        valid_meta = pd.read_csv(preprocessing_files["valid_meta"])
        feature_names = pd.read_csv(preprocessing_files["feature_names"])
        manifest = json.loads(preprocessing_files["manifest"].read_text(encoding="utf-8"))
        baseline_validation_scores = pd.read_csv(baseline_files["validation_scores"])
        baseline_model = joblib.load(baseline_files["model"])
        baseline_metrics = json.loads(baseline_files["metrics"].read_text(encoding="utf-8"))
        advanced_validation_scores = pd.read_csv(advanced_files["validation_scores"])
        advanced_model = joblib.load(advanced_files["model"])
        advanced_metrics = json.loads(advanced_files["metrics"].read_text(encoding="utf-8"))

        feature_name_column = (
            "feature_name" if "feature_name" in feature_names.columns else feature_names.columns[0]
        )
        feature_name_list = feature_names[feature_name_column].astype(str).tolist()
        raw_feature_candidates = manifest.get("feature_groups", {}).get("numeric", []) + manifest.get(
            "feature_groups", {}
        ).get("categorical", [])

        if settings.id_column not in valid_meta.columns:
            validation_errors.append(f"{feature_set_name}: valid_meta 缺少主键列")
        if settings.target_column not in valid_meta.columns:
            validation_errors.append(f"{feature_set_name}: valid_meta 缺少目标列")
        if X_valid.shape[0] != len(valid_meta):
            validation_errors.append(f"{feature_set_name}: X_valid 行数与 valid_meta 不一致")
        if len(feature_name_list) != X_valid.shape[1]:
            validation_errors.append(f"{feature_set_name}: feature_names 条数与 X_valid 列数不一致")
        if not raw_feature_candidates:
            validation_errors.append(f"{feature_set_name}: manifest 未提供原始字段分组")

        baseline_required_columns = {settings.id_column, "predicted_pd"}
        advanced_required_columns = {
            settings.id_column,
            "advanced_predicted_pd",
            "logistic_predicted_pd",
            "pd_delta_advanced_minus_logistic",
        }
        missing_baseline_columns = baseline_required_columns - set(baseline_validation_scores.columns)
        missing_advanced_columns = advanced_required_columns - set(advanced_validation_scores.columns)
        if missing_baseline_columns:
            validation_errors.append(
                f"{feature_set_name}: baseline validation_scores 缺少列 {sorted(missing_baseline_columns)}"
            )
        if missing_advanced_columns:
            validation_errors.append(
                f"{feature_set_name}: advanced validation_scores 缺少列 {sorted(missing_advanced_columns)}"
            )
        if valid_meta[settings.id_column].duplicated().any():
            validation_errors.append(f"{feature_set_name}: valid_meta 主键不是唯一值")
        if baseline_validation_scores[settings.id_column].duplicated().any():
            validation_errors.append(f"{feature_set_name}: baseline validation_scores 主键不是唯一值")
        if advanced_validation_scores[settings.id_column].duplicated().any():
            validation_errors.append(f"{feature_set_name}: advanced validation_scores 主键不是唯一值")

        review_base = valid_meta[[settings.id_column, settings.target_column]].merge(
            review_source_frame,
            on=settings.id_column,
            how="left",
            indicator=True,
        )
        if (review_base["_merge"] != "both").any():
            validation_errors.append(
                f"{feature_set_name}: application_train fairness / governance 字段无法覆盖全部 validation 样本"
            )
        review_base = review_base.drop(columns=["_merge"])

        logistic_review = review_base.merge(
            baseline_validation_scores[[settings.id_column, "predicted_pd"]],
            on=settings.id_column,
            how="left",
        )
        advanced_review = review_base.merge(
            advanced_validation_scores[
                [
                    settings.id_column,
                    "advanced_predicted_pd",
                    "logistic_predicted_pd",
                    "pd_delta_advanced_minus_logistic",
                ]
            ],
            on=settings.id_column,
            how="left",
        ).rename(columns={"advanced_predicted_pd": "predicted_pd"})

        if logistic_review["predicted_pd"].isna().any():
            validation_errors.append(f"{feature_set_name}: baseline 分数无法覆盖全部 validation 样本")
        if advanced_review["predicted_pd"].isna().any():
            validation_errors.append(f"{feature_set_name}: advanced 分数无法覆盖全部 validation 样本")

        sorted_valid_meta = valid_meta[[settings.id_column, settings.target_column]].sort_values(
            settings.id_column
        ).reset_index(drop=True)
        if reference_valid_meta is None:
            reference_valid_meta = sorted_valid_meta
        elif not reference_valid_meta.equals(sorted_valid_meta):
            validation_errors.append(f"{feature_set_name}: valid_meta 与参考 feature set 的主键/目标不一致")

        advanced_backend = advanced_metrics.get("backend")
        if advanced_backend_reference is None:
            advanced_backend_reference = advanced_backend
        elif advanced_backend_reference != advanced_backend:
            validation_errors.append(
                f"{feature_set_name}: advanced backend 与参考 feature set 不一致（{advanced_backend_reference} vs {advanced_backend}）"
            )

        artifacts_by_set[feature_set_name] = {
            "X_valid": X_valid,
            "valid_meta": valid_meta,
            "feature_names": feature_name_list,
            "raw_feature_candidates": raw_feature_candidates,
            "manifest": manifest,
            "baseline_model": baseline_model,
            "baseline_metrics": baseline_metrics,
            "advanced_model": advanced_model,
            "advanced_metrics": advanced_metrics,
            "advanced_backend": advanced_backend,
        }
        review_frames_by_set[feature_set_name] = {
            "logistic": logistic_review,
            "advanced": advanced_review,
        }
        load_summary_rows.extend(
            [
                {
                    "feature_set": feature_set_name,
                    "artifact": "X_valid",
                    "shape": X_valid.shape,
                    "density": _matrix_density(X_valid),
                    "advanced_backend": advanced_backend,
                },
                {
                    "feature_set": feature_set_name,
                    "artifact": "logistic_review",
                    "shape": logistic_review.shape,
                    "density": None,
                    "advanced_backend": advanced_backend,
                },
                {
                    "feature_set": feature_set_name,
                    "artifact": "advanced_review",
                    "shape": advanced_review.shape,
                    "density": None,
                    "advanced_backend": advanced_backend,
                },
            ]
        )

    phase5_load_ok = not validation_errors
    display(pd.DataFrame(load_summary_rows))
    if validation_errors:
        display(pd.DataFrame({"validation_error": validation_errors}))
        print("Phase 5 stopped because the saved preprocessing / modeling artifacts failed validation.")
    else:
        print("Phase 5 load validation passed for both feature sets.")



## 4. 候选模型与对照模型选择

默认规则固定为：先按 `ROC-AUC`、再按 `AP`、再按模型家族优先级、最后按 `traditional_plus_proxy` 优先级排序。之后再为同一模型家族解析 `matched_core_comparator`、`matched_proxy_comparator` 和 `proxy_audit_pair`。


In [ ]:

if not PHASE_5_READY:
    print("候选模型选择已跳过，因为 Phase 5 依赖产物不完整。")
elif not phase5_load_ok:
    print("候选模型选择已跳过，因为加载校验未通过。")
else:
    selection_payload = {}
    candidate_model_selection_payload = {}

    required_comparison_columns = {
        "feature_set",
        "model_family",
        "model_label",
        "roc_auc",
        "average_precision",
    }
    missing_comparison_columns = required_comparison_columns - set(phase4_comparison_metrics.columns)
    if missing_comparison_columns:
        print(
            f"Phase 5 stopped because Phase 4 comparison_metrics 缺少列 {sorted(missing_comparison_columns)}"
        )
        phase5_selection_ok = False
    else:
        selection_payload = select_candidate_payload(phase4_comparison_metrics)
        ranked_phase4_comparison = selection_payload.pop("ranked_comparison")
        candidate_model_selection_payload = {
            key: value for key, value in selection_payload.items()
        }
        phase5_selection_ok = (
            candidate_model_selection_payload.get("candidate_model") is not None
            and candidate_model_selection_payload.get("matched_core_comparator") is not None
            and candidate_model_selection_payload.get("matched_proxy_comparator") is not None
        )

        display(ranked_phase4_comparison)
        display(pd.DataFrame([candidate_model_selection_payload["candidate_model"]]))
        display(pd.DataFrame([candidate_model_selection_payload["matched_core_comparator"]]))
        display(pd.DataFrame([candidate_model_selection_payload["matched_proxy_comparator"]]))
        print(f"Proxy audit pair family: {candidate_model_selection_payload['proxy_audit_pair_family']}")
        print(f"Proxy audit pair reason: {candidate_model_selection_payload['proxy_audit_pair_reason']}")



## 5. Explainability

这里会对 `proxy_audit_pair` 的两套 feature set 同步计算解释结果，并保留候选模型与 `matched_core_comparator` 的对照视角：

- transformed feature 级别全局贡献
- raw feature 聚合贡献
- proxy family 聚合贡献
- 代表样本的本地解释（仅在 `xgboost pred_contribs` 可用时生成）


In [ ]:

if not PHASE_5_READY:
    print("Explainability 步骤已跳过，因为 Phase 5 依赖产物不完整。")
elif not phase5_load_ok:
    print("Explainability 步骤已跳过，因为加载校验未通过。")
elif not phase5_selection_ok:
    print("Explainability 步骤已跳过，因为候选模型选择未完成。")
else:
    explainability_outputs_by_set = {}
    local_case_outputs_by_set = {}
    local_case_selection = pd.DataFrame()
    proxy_uplift_summary = pd.DataFrame()
    explainability_summary_rows = []

    proxy_audit_pair_family = candidate_model_selection_payload["proxy_audit_pair_family"]
    candidate_feature_set = candidate_model_selection_payload["candidate_model"]["feature_set"]
    pair_backend = (
        artifacts_by_set[FEATURE_SET_NAMES[0]]["advanced_backend"]
        if proxy_audit_pair_family == "advanced"
        else "logistic_regression_baseline"
    )
    explainability_method = (
        "xgboost_pred_contribs"
        if proxy_audit_pair_family == "advanced"
        and pair_backend == "xgboost"
        and runtime_status["xgboost_contrib_ok"]
        else "permutation_importance"
    )

    for feature_set_name in FEATURE_SET_NAMES:
        feature_artifacts = artifacts_by_set[feature_set_name]
        review_frame = review_frames_by_set[feature_set_name][proxy_audit_pair_family]
        model = (
            feature_artifacts["advanced_model"]
            if proxy_audit_pair_family == "advanced"
            else feature_artifacts["baseline_model"]
        )
        if explainability_method == "xgboost_pred_contribs":
            summary = compute_xgboost_contribution_summary(
                model=model,
                X_matrix=feature_artifacts["X_valid"],
                feature_names=feature_artifacts["feature_names"],
                raw_feature_candidates=feature_artifacts["raw_feature_candidates"],
                sample_size=EXPLAIN_SAMPLE_SIZE,
                random_state=RANDOM_STATE,
            )
        else:
            summary = compute_permutation_importance_summary(
                model=model,
                X_matrix=feature_artifacts["X_valid"],
                y_true=review_frame[settings.target_column].to_numpy(),
                feature_names=feature_artifacts["feature_names"],
                raw_feature_candidates=feature_artifacts["raw_feature_candidates"],
                sample_size=EXPLAIN_SAMPLE_SIZE,
                scoring="roc_auc",
                n_repeats=5,
                random_state=RANDOM_STATE,
            )

        summary["feature_set"] = feature_set_name
        summary["model_family"] = proxy_audit_pair_family
        summary["backend"] = pair_backend
        explainability_outputs_by_set[feature_set_name] = summary
        explainability_summary_rows.append(
            {
                "feature_set": feature_set_name,
                "model_family": proxy_audit_pair_family,
                "backend": pair_backend,
                "explainability_method": summary["explainability_method"],
                "sample_rows": len(summary["sample_indices"]),
                "raw_feature_count": len(summary["raw_feature_contributions"]),
                "proxy_family_count": len(summary["proxy_family_contributions"]),
            }
        )

    candidate_review_frame = review_frames_by_set[candidate_feature_set][proxy_audit_pair_family].copy()
    local_case_selection = select_local_explanation_rows(
        candidate_review_frame,
        score_column="predicted_pd",
        target_column=settings.target_column,
        threshold=OPERATING_THRESHOLD,
        id_column=settings.id_column,
        num_cases=LOCAL_CASES,
    ).rename(columns={"predicted_pd": "candidate_predicted_pd"})

    if explainability_method == "xgboost_pred_contribs":
        for feature_set_name in FEATURE_SET_NAMES:
            feature_review_scores = review_frames_by_set[feature_set_name][proxy_audit_pair_family][
                [settings.id_column, "predicted_pd"]
            ].rename(columns={"predicted_pd": "feature_set_predicted_pd"})
            local_selection_for_set = local_case_selection.merge(
                feature_review_scores,
                on=settings.id_column,
                how="left",
            )
            local_case_outputs_by_set[feature_set_name] = compute_xgboost_local_explanations(
                model=(
                    artifacts_by_set[feature_set_name]["advanced_model"]
                    if proxy_audit_pair_family == "advanced"
                    else artifacts_by_set[feature_set_name]["baseline_model"]
                ),
                X_matrix=artifacts_by_set[feature_set_name]["X_valid"],
                feature_names=artifacts_by_set[feature_set_name]["feature_names"],
                selected_rows=local_selection_for_set,
                raw_feature_candidates=artifacts_by_set[feature_set_name]["raw_feature_candidates"],
                top_n_features=min(TOP_N_FEATURES, 10),
            )
    else:
        for feature_set_name in FEATURE_SET_NAMES:
            local_case_outputs_by_set[feature_set_name] = pd.DataFrame()

    core_proxy_families = explainability_outputs_by_set["traditional_core"][
        "proxy_family_contributions"
    ][["proxy_family", "mean_abs_contribution", "mean_contribution"]].rename(
        columns={
            "mean_abs_contribution": "traditional_core_mean_abs_contribution",
            "mean_contribution": "traditional_core_mean_contribution",
        }
    )
    proxy_proxy_families = explainability_outputs_by_set["traditional_plus_proxy"][
        "proxy_family_contributions"
    ][["proxy_family", "mean_abs_contribution", "mean_contribution"]].rename(
        columns={
            "mean_abs_contribution": "traditional_plus_proxy_mean_abs_contribution",
            "mean_contribution": "traditional_plus_proxy_mean_contribution",
        }
    )
    proxy_uplift_summary = core_proxy_families.merge(
        proxy_proxy_families,
        on="proxy_family",
        how="outer",
    ).fillna(0.0)
    proxy_uplift_summary["proxy_audit_pair_family"] = proxy_audit_pair_family
    proxy_uplift_summary["mean_abs_contribution_delta_proxy_minus_core"] = (
        proxy_uplift_summary["traditional_plus_proxy_mean_abs_contribution"]
        - proxy_uplift_summary["traditional_core_mean_abs_contribution"]
    )
    proxy_uplift_summary["mean_contribution_delta_proxy_minus_core"] = (
        proxy_uplift_summary["traditional_plus_proxy_mean_contribution"]
        - proxy_uplift_summary["traditional_core_mean_contribution"]
    )
    proxy_uplift_summary = proxy_uplift_summary.sort_values(
        "mean_abs_contribution_delta_proxy_minus_core",
        ascending=False,
    ).reset_index(drop=True)

    phase5_explainability_ok = True
    display(pd.DataFrame(explainability_summary_rows))
    display(local_case_selection)
    display(proxy_uplift_summary)
    print(f"Explainability method: {explainability_method}")



## 6. Fairness / Governance grouped review

这里基于候选模型分数构造统一的 validation review frame，并对以下分组做 grouped operational summary：

- `age_band`
- `family_status_group`
- `region_rating_group`
- `city_work_mismatch_group`
- `organization_group`
- `occupation_group`

所有 grouped 指标都只反映 validation split 上的分析结果，不应直接当成生产结论。


In [ ]:

if not PHASE_5_READY:
    print("Fairness / governance 步骤已跳过，因为 Phase 5 依赖产物不完整。")
elif not phase5_explainability_ok:
    print("Fairness / governance 步骤已跳过，因为 explainability 步骤未完成。")
else:
    pair_family = candidate_model_selection_payload["proxy_audit_pair_family"]
    core_score_column = f"{pair_family}_traditional_core_predicted_pd"
    proxy_score_column = f"{pair_family}_traditional_plus_proxy_predicted_pd"
    candidate_feature_set = candidate_model_selection_payload["candidate_model"]["feature_set"]

    core_review = review_frames_by_set["traditional_core"][pair_family][
        [
            settings.id_column,
            settings.target_column,
            "predicted_pd",
            "DAYS_BIRTH",
            "NAME_FAMILY_STATUS",
            "REGION_RATING_CLIENT",
            "REG_CITY_NOT_WORK_CITY",
            "ORGANIZATION_TYPE",
            "OCCUPATION_TYPE",
        ]
    ].rename(columns={"predicted_pd": core_score_column})
    proxy_review = review_frames_by_set["traditional_plus_proxy"][pair_family][
        [settings.id_column, "predicted_pd"]
    ].rename(columns={"predicted_pd": proxy_score_column})

    validation_review_frame = core_review.merge(
        proxy_review,
        on=settings.id_column,
        how="inner",
    )
    validation_review_frame["proxy_minus_core_predicted_pd_delta"] = (
        validation_review_frame[proxy_score_column] - validation_review_frame[core_score_column]
    )
    validation_review_frame["candidate_predicted_pd"] = validation_review_frame[
        proxy_score_column if candidate_feature_set == "traditional_plus_proxy" else core_score_column
    ]
    validation_review_frame["candidate_feature_set"] = candidate_feature_set
    validation_review_frame["candidate_model_family"] = candidate_model_selection_payload["candidate_model"][
        "model_family"
    ]

    validation_review_frame["age_band"] = derive_age_band_from_days_birth(
        validation_review_frame["DAYS_BIRTH"]
    )
    validation_review_frame["family_status_group"] = (
        validation_review_frame["NAME_FAMILY_STATUS"].astype("object").fillna("Missing")
    )
    validation_review_frame["region_rating_group"] = (
        validation_review_frame["REGION_RATING_CLIENT"].astype("object").fillna("Missing")
    )
    validation_review_frame["city_work_mismatch_group"] = (
        validation_review_frame["REG_CITY_NOT_WORK_CITY"].astype("object").fillna("Missing")
    )
    validation_review_frame["organization_group"] = collapse_rare_categories(
        validation_review_frame["ORGANIZATION_TYPE"],
        top_n=TOP_N_CATEGORIES,
    )
    validation_review_frame["occupation_group"] = collapse_rare_categories(
        validation_review_frame["OCCUPATION_TYPE"],
        top_n=TOP_N_CATEGORIES,
    )

    group_fairness_summary = build_group_fairness_summary(
        validation_review_frame,
        target_column=settings.target_column,
        score_column="candidate_predicted_pd",
        group_specs=GROUP_SPECS,
        threshold=OPERATING_THRESHOLD,
        top_n_categories=TOP_N_CATEGORIES,
    )
    group_gap_summary = build_group_gap_summary(group_fairness_summary)
    phase5_fairness_ok = not group_fairness_summary.empty

    display(group_gap_summary)
    display(group_fairness_summary.head(30))



## 7. 可视化、落盘与 checkpoint

最后一步会把 explainability、proxy uplift、grouped fairness review 和 local case 统一出图并落盘：

- `data/processed/xai_fairness/`
- `data/processed/xai_fairness/<feature_set>/`
- `reports/figures/phase5_*.png`

默认会生成不少于 20 张图，便于后续人工分析与报告摘录。


In [ ]:

if not PHASE_5_READY:
    print("可视化与落盘已跳过，因为 Phase 5 依赖产物不完整。")
elif not phase5_fairness_ok:
    print("可视化与落盘已跳过，因为 grouped fairness review 未完成。")
else:
    PHASE_5_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    figure_paths = {}

    candidate_feature_set = candidate_model_selection_payload["candidate_model"]["feature_set"]
    candidate_summary = explainability_outputs_by_set[candidate_feature_set]
    matched_core_summary = explainability_outputs_by_set["traditional_core"]
    proxy_summary = explainability_outputs_by_set["traditional_plus_proxy"]

    figure_paths["candidate_global_feature_contributions"] = (
        FIGURE_OUTPUT_DIR / f"phase5_{candidate_feature_set}_candidate_global_feature_contributions.png"
    )
    save_horizontal_bar_chart(
        candidate_summary["global_feature_contributions"],
        value_column="mean_abs_contribution",
        label_column="transformed_feature_name",
        title=f"Phase 5 Candidate Global Contributions ({candidate_feature_set})",
        path=figure_paths["candidate_global_feature_contributions"],
        top_n=TOP_N_FEATURES,
    )

    figure_paths["candidate_proxy_family_contributions"] = (
        FIGURE_OUTPUT_DIR / f"phase5_{candidate_feature_set}_candidate_proxy_family_contributions.png"
    )
    save_horizontal_bar_chart(
        candidate_summary["proxy_family_contributions"],
        value_column="mean_abs_contribution",
        label_column="proxy_family",
        title=f"Phase 5 Candidate Proxy Family Contributions ({candidate_feature_set})",
        path=figure_paths["candidate_proxy_family_contributions"],
    )

    figure_paths["matched_core_global_feature_contributions"] = (
        FIGURE_OUTPUT_DIR / "phase5_traditional_core_matched_core_global_feature_contributions.png"
    )
    save_horizontal_bar_chart(
        matched_core_summary["global_feature_contributions"],
        value_column="mean_abs_contribution",
        label_column="transformed_feature_name",
        title="Phase 5 Matched Core Comparator Global Contributions",
        path=figure_paths["matched_core_global_feature_contributions"],
        top_n=TOP_N_FEATURES,
    )

    figure_paths["matched_core_proxy_family_contributions"] = (
        FIGURE_OUTPUT_DIR / "phase5_traditional_core_matched_core_proxy_family_contributions.png"
    )
    save_horizontal_bar_chart(
        matched_core_summary["proxy_family_contributions"],
        value_column="mean_abs_contribution",
        label_column="proxy_family",
        title="Phase 5 Matched Core Comparator Proxy Family Contributions",
        path=figure_paths["matched_core_proxy_family_contributions"],
    )

    figure_paths["proxy_family_uplift_delta"] = (
        FIGURE_OUTPUT_DIR / "phase5_proxy_family_uplift_delta.png"
    )
    save_horizontal_bar_chart(
        proxy_uplift_summary,
        value_column="mean_abs_contribution_delta_proxy_minus_core",
        label_column="proxy_family",
        title="Phase 5 Proxy Family Uplift Delta (Proxy - Core)",
        path=figure_paths["proxy_family_uplift_delta"],
    )

    figure_paths["proxy_minus_core_score_delta_histogram"] = (
        FIGURE_OUTPUT_DIR / "phase5_proxy_minus_core_score_delta_histogram.png"
    )
    save_histogram(
        validation_review_frame,
        column="proxy_minus_core_predicted_pd_delta",
        title="Phase 5 Proxy vs Core Predicted PD Delta Distribution",
        path=figure_paths["proxy_minus_core_score_delta_histogram"],
    )

    group_figure_specs = [
        ("age_band", "actual_default_rate", "Age Band Actual Default Rate", "phase5_age_band_actual_default_rate.png"),
        ("age_band", "mean_predicted_pd", "Age Band Mean Predicted PD", "phase5_age_band_mean_predicted_pd.png"),
        ("age_band", "approval_rate", "Age Band Approval Rate", "phase5_age_band_approval_rate.png"),
        ("family_status_group", "actual_default_rate", "Family Status Actual Default Rate", "phase5_family_status_actual_default_rate.png"),
        ("family_status_group", "mean_predicted_pd", "Family Status Mean Predicted PD", "phase5_family_status_mean_predicted_pd.png"),
        ("family_status_group", "approval_rate", "Family Status Approval Rate", "phase5_family_status_approval_rate.png"),
        ("region_rating_group", "actual_default_rate", "Region Rating Actual Default Rate", "phase5_region_rating_actual_default_rate.png"),
        ("region_rating_group", "approval_rate", "Region Rating Approval Rate", "phase5_region_rating_approval_rate.png"),
        ("city_work_mismatch_group", "actual_default_rate", "City / Work Mismatch Actual Default Rate", "phase5_city_work_mismatch_actual_default_rate.png"),
        ("organization_group", "actual_default_rate", "Organization Group Actual Default Rate", "phase5_organization_group_actual_default_rate.png"),
        ("occupation_group", "actual_default_rate", "Occupation Group Actual Default Rate", "phase5_occupation_group_actual_default_rate.png"),
    ]
    for protected_attribute, metric_column, title, filename in group_figure_specs:
        figure_key = filename.replace(".png", "")
        figure_paths[figure_key] = FIGURE_OUTPUT_DIR / filename
        save_group_metric_chart(
            group_fairness_summary,
            protected_attribute=protected_attribute,
            metric_column=metric_column,
            title=title,
            path=figure_paths[figure_key],
        )

    candidate_local_frame = local_case_outputs_by_set.get(candidate_feature_set, pd.DataFrame())
    case_roles = local_case_selection["case_role"].tolist()[:LOCAL_CASES]
    for case_number in range(LOCAL_CASES):
        case_role = case_roles[case_number] if case_number < len(case_roles) else f"placeholder_case_{case_number + 1}"
        figure_key = f"local_case_{case_number + 1}"
        figure_paths[figure_key] = FIGURE_OUTPUT_DIR / f"phase5_{candidate_feature_set}_{figure_key}.png"
        save_local_case_chart(
            candidate_local_frame,
            case_role=case_role,
            title=f"Phase 5 Local Explanation #{case_number + 1} ({case_role})",
            path=figure_paths[figure_key],
        )

    for feature_set_name in FEATURE_SET_NAMES:
        output_dir = PHASE_5_OUTPUT_ROOT / feature_set_name
        output_dir.mkdir(parents=True, exist_ok=True)
        explainability_outputs_by_set[feature_set_name]["global_feature_contributions"].to_csv(
            output_dir / "global_feature_contributions.csv",
            index=False,
        )
        explainability_outputs_by_set[feature_set_name]["raw_feature_contributions"].to_csv(
            output_dir / "raw_feature_contributions.csv",
            index=False,
        )
        explainability_outputs_by_set[feature_set_name]["proxy_family_contributions"].to_csv(
            output_dir / "proxy_family_contributions.csv",
            index=False,
        )
        local_case_outputs_by_set.get(feature_set_name, pd.DataFrame()).to_csv(
            output_dir / "local_case_explanations.csv",
            index=False,
        )
        metrics_payload = {
            "feature_set": feature_set_name,
            "model_family": candidate_model_selection_payload["proxy_audit_pair_family"],
            "backend": explainability_outputs_by_set[feature_set_name]["backend"],
            "explainability_method": explainability_outputs_by_set[feature_set_name]["explainability_method"],
            "sample_rows": len(explainability_outputs_by_set[feature_set_name]["sample_indices"]),
            "runtime_status": runtime_status,
            "local_case_count": int(
                local_case_outputs_by_set.get(feature_set_name, pd.DataFrame())["case_role"].nunique()
                if not local_case_outputs_by_set.get(feature_set_name, pd.DataFrame()).empty
                else 0
            ),
            "figure_paths": {
                key: str(path)
                for key, path in figure_paths.items()
                if feature_set_name in str(path) or "phase5_" in path.name
            },
            "governance_note": "Grouped diagnostics only. Not a production-ready fairness audit.",
        }
        (output_dir / "metrics.json").write_text(
            json.dumps(_to_builtin(metrics_payload), indent=2, ensure_ascii=False),
            encoding="utf-8",
        )

    (PHASE_5_OUTPUT_ROOT / "candidate_model_selection.json").write_text(
        json.dumps(_to_builtin(candidate_model_selection_payload), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    proxy_uplift_summary.to_csv(PHASE_5_OUTPUT_ROOT / "proxy_uplift_summary.csv", index=False)
    group_fairness_summary.to_csv(PHASE_5_OUTPUT_ROOT / "group_fairness_summary.csv", index=False)
    validation_review_frame.to_csv(PHASE_5_OUTPUT_ROOT / "validation_review_frame.csv", index=False)

    if not group_gap_summary.empty:
        max_gap_row = group_gap_summary.sort_values("max_approval_rate_gap", ascending=False).iloc[0]
        group_gap_message = (
            f"Largest approval-rate gap currently appears in {max_gap_row['protected_attribute']}."
        )
    else:
        group_gap_message = "Grouped approval-rate gaps were not computed."

    top_proxy_families = proxy_uplift_summary.head(3)["proxy_family"].tolist()
    business_conclusion_template = {
        "proxy_uplift_statement": (
            "Proxy uplift review should start with "
            + (", ".join(top_proxy_families) if top_proxy_families else "no material proxy families")
            + "."
        ),
        "group_outcome_statement": group_gap_message,
        "governance_note": "Current output is a grouped diagnostic and must not be described as a production-ready fairness audit.",
    }
    summary_payload = {
        "candidate_model_selection": candidate_model_selection_payload,
        "runtime_status": runtime_status,
        "proxy_audit_pair_family": candidate_model_selection_payload["proxy_audit_pair_family"],
        "explainability_method_by_feature_set": {
            feature_set_name: explainability_outputs_by_set[feature_set_name]["explainability_method"]
            for feature_set_name in FEATURE_SET_NAMES
        },
        "group_gap_summary": group_gap_summary.to_dict(orient="records"),
        "top_proxy_uplift_rows": proxy_uplift_summary.head(10).to_dict(orient="records"),
        "figure_paths": {key: str(path) for key, path in figure_paths.items()},
        "business_conclusion_template": business_conclusion_template,
        "governance_limitations": [
            "This notebook reports grouped validation diagnostics only.",
            "Age band and family status are baseline review groups, not a completed legal or compliance standard.",
            "Proxy-sensitive uplift should not be treated as production-ready gain until governance review is complete.",
        ],
    }
    (PHASE_5_OUTPUT_ROOT / "summary.json").write_text(
        json.dumps(_to_builtin(summary_payload), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    saved_outputs = [
        {"artifact": "candidate_model_selection", "path": str(PHASE_5_OUTPUT_ROOT / "candidate_model_selection.json")},
        {"artifact": "proxy_uplift_summary", "path": str(PHASE_5_OUTPUT_ROOT / "proxy_uplift_summary.csv")},
        {"artifact": "group_fairness_summary", "path": str(PHASE_5_OUTPUT_ROOT / "group_fairness_summary.csv")},
        {"artifact": "validation_review_frame", "path": str(PHASE_5_OUTPUT_ROOT / "validation_review_frame.csv")},
        {"artifact": "summary", "path": str(PHASE_5_OUTPUT_ROOT / "summary.json")},
    ]
    display(pd.DataFrame(saved_outputs))
    print(f"Saved figure count: {len(figure_paths)}")



## Checkpoint

到这里为止，`Phase 5` 已经把候选模型解释、proxy uplift 与 grouped fairness / governance review 串成一个 notebook-friendly 流程。接下来进入 `Phase 6` 时，必须把这里识别出的 proxy-sensitive uplift 与 grouped outcome 差异一起带入 cutoff / scorecard 讨论，而不是只看 AUC 提升。


In [ ]:

if not PHASE_5_READY:
    print("Checkpoint: Phase 5 仍被阻断，请先补齐双 feature-set 的 Phase 2 / 3 / 4 标准产物。")
elif not phase5_load_ok:
    print("Checkpoint: Phase 5 仍停在加载校验，请先修复 artifact 一致性问题。")
elif not phase5_selection_ok:
    print("Checkpoint: Phase 5 仍停在候选模型选择，请先修复 Phase 4 comparison_metrics。")
elif not phase5_explainability_ok:
    print("Checkpoint: Phase 5 仍停在 explainability，请先修复 explainability runtime 或 model artifact。")
elif not phase5_fairness_ok:
    print("Checkpoint: Phase 5 仍停在 grouped fairness review，请先修复 validation review frame。")
else:
    print("Phase 5 XAI + fairness review complete.")
    print(
        f"Candidate model: {candidate_model_selection_payload['candidate_model']['model_label']}"
    )
    print(
        f"Proxy audit pair family: {candidate_model_selection_payload['proxy_audit_pair_family']}"
    )
    print(f"Generated figures: {len(figure_paths)}")
    display(proxy_uplift_summary.head(10))
    display(group_gap_summary)
    print(
        "治理提醒：当前结果只代表 grouped diagnostic，不代表 production-ready fairness audit。"
    )
